<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/02_validacion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Validación del dataset

Auditoría de archivos, etiquetas YOLO, geometría, duplicados y split. **No modifica
imágenes, etiquetas ni la división compartida.** Ejecutar todo.

- **Error:** archivo, anotación o separación que necesita corregirse o justificarse antes de entrenar.
- **Advertencia:** caso para revisar; no se elimina ni transforma automáticamente.
- IDs esperados: `0, 1, 2`. En `solarpanels_hd` son niveles de confianza; ver `docs/criterios_datos.md`.
- Una etiqueta vacía es negativa; una faltante o inválida queda como desconocida.
- Tolerancia de bordes: `1e-6` normalizado por redondeo (0.000256 px en estas imágenes).
- Cajas menores a 3 px en alguna dimensión se señalan para revisión, sin filtrarlas.


In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

from pdi_tp import entorno, datos, resultados, split
rutas = entorno.preparar()


## Ejecutar y exportar

Se revisan todas las imágenes y etiquetas, aunque alguna esté dañada. Los duplicados
se comparan por SHA-256 de los archivos y de los píxeles RGB con sus dimensiones.
Esto detecta copias exactas con distinta compresión o modo de color; **no detecta
imágenes casi iguales**, rotaciones o recortes superpuestos.

La comprobación espacial aplica la heurística ya existente en `pdi_tp/split.py`;
no prueba por sí sola que no haya escenas relacionadas entre grupos distintos.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pdi_tp import validacion

informe = validacion.validar(rutas, clases=(0, 1, 2), minimo_px=3, tolerancia=1e-6)
salida = rutas.analisis / "validacion_datos"
informe.guardar(salida)
print(f"Reporte completo: {salida}")
display(informe.controles)
display(pd.Series({k: v for k, v in informe.resumen.items() if k != "parametros"}, name="cantidad").to_frame())


## Imágenes, etiquetas y clases

El modo paleta `P` se registra, pero no es un error: el lector compartido convierte
las imágenes a RGB. Las cajas de la tabla siguiente son filas de cinco valores
numéricos finitos; los problemas de lectura están detallados en las incidencias.


In [ ]:
display(informe.imagenes.groupby(["ancho", "alto", "modo"], dropna=False).size().rename("imagenes").to_frame())
display(informe.cajas.groupby(["clase", "valida"], dropna=False).size().rename("cajas").to_frame())
display(informe.imagenes.groupby(["split", "fuente"], dropna=False).agg(
    imagenes=("archivo", "size"), etiquetas_validas=("etiqueta_valida", "sum")))


## Tabla de incidencias

Cada fila indica un problema y, si corresponde, la línea de la etiqueta original.
Una caja puede tener más de una incidencia; el total de incidencias no es el total
de cajas afectadas. Los CSV contienen todos los registros, sin el límite de esta vista.


In [ ]:
if informe.incidencias.empty:
    print("No se encontraron incidencias.")
else:
    display(informe.incidencias.groupby(["nivel", "control"]).agg(
        incidencias=("archivo", "size"), archivos=("archivo", "nunique")))
    with pd.option_context("display.max_colwidth", 140):
        display(informe.incidencias.head(50))


## Ejemplos de errores y cajas pequeñas

La tabla da el nombre completo. En los gráficos `L` es la línea de la etiqueta y `C`
el ID de clase. Las cajas se dibujan sin corregir ni recortar sus coordenadas.


In [ ]:
candidatos = informe.incidencias.loc[
    (informe.incidencias["nivel"] == "error") & informe.incidencias["archivo"].isin(informe.imagenes["archivo"]),
    "archivo"].drop_duplicates().head(8).tolist()
display(informe.imagenes.loc[informe.imagenes["archivo"].isin(candidatos), ["archivo", "split", "etiqueta_valida"]])
fig = validacion.dibujar_ejemplos(informe, rutas, candidatos)
if fig is not None:
    fig.savefig(salida / "ejemplos_errores.png", dpi=120, bbox_inches="tight")
    display(fig)
    plt.close(fig)

pequenas = informe.cajas.loc[informe.cajas["pequena"]].sort_values(["w_px", "h_px"])
display(pequenas[["archivo", "linea", "w_px", "h_px", "valida"]].head(12))
fig = validacion.dibujar_ejemplos(informe, rutas, pequenas["archivo"], maximo=8)
if fig is not None:
    fig.savefig(salida / "ejemplos_cajas_pequenas.png", dpi=120, bbox_inches="tight")
    display(fig)
    plt.close(fig)


## Duplicados exactos

Un mismo grupo puede aparecer en la comparación por bytes y por píxeles: no sumar
ambos como si fueran grupos diferentes. Un duplicado que cruza train/val se marca
como error. Si tiene anotaciones distintas, también se registra una advertencia.


In [ ]:
display(informe.duplicados)
if not informe.duplicados.empty:
    nombres = [n for grupo in informe.duplicados.loc[informe.duplicados["tipo"] == "pixeles_rgb", "archivos"]
               for n in grupo.split(" | ")]
    fig = validacion.dibujar_ejemplos(informe, rutas, nombres, maximo=8)
    if fig is not None:
        fig.savefig(salida / "ejemplos_duplicados.png", dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)


## Clases minoritarias para revisión visual

Esta galería ayuda a revisar los IDs 1 y 2. Su significado en `solarpanels_hd` se cotejó con la fuente original: confianza moderada y baja. La fusión en una clase "panel" para el futuro derivado está documentada en `docs/criterios_datos.md`.


In [ ]:
raras = informe.cajas.loc[informe.cajas["clase"].isin([1, 2])]
display(raras[["archivo", "linea", "clase", "w_px", "h_px", "valida"]])
fig = validacion.dibujar_ejemplos(informe, rutas, raras["archivo"], maximo=8)
if fig is not None:
    fig.savefig(salida / "ejemplos_clases_minoritarias.png", dpi=120, bbox_inches="tight")
    display(fig)
    plt.close(fig)


## Qué decidir después

1. Revisar los errores y sus líneas antes de preparar datos de entrenamiento.
2. Consultar la definición de clases y decisiones en `docs/criterios_datos.md`.
3. Conservar cajas pequeñas y originales; excluir la caja nula solo al preparar el derivado.
4. Si se requieren cambios del split, decidirlos como equipo y versionar una nueva división.

No aplicar correcciones silenciosas. La validación estructural no comprueba por sí
sola que todas las anotaciones sean semánticamente correctas o estén completas.
